In [ ]:
!pip install --upgrade scikit-learn imbalanced-learn

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# --- Install necessary libraries ---
!pip install -q transformers datasets accelerate scikit-learn torch

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    EarlyStoppingCallback # <--- IMPORT ADDED
)
from datasets import Dataset

# --- Configuration ---
DATA_PATH = "/kaggle/input/github-issues-csv/github_issues.csv"
OUTPUT_DIR = "/kaggle/working"
MODEL_NAME = "distilbert-base-uncased" 
RANDOM_STATE = 42

print("=========================================================")
print("  STARTING DEEP LEARNING: 3-CLASS FINE-TUNING (10 EPOCHS)  ")
print("=========================================================")

# 1. Load and Prepare Data
print("Loading data...")
df = pd.read_csv(DATA_PATH)

# Deduplication (Best Practice)
if 'id' in df.columns:
    df = df.drop_duplicates(subset=['id'], keep='last')
else:
    df = df.drop_duplicates(subset=['title', 'body'], keep='last')

df["title"] = df["title"].fillna("")
df["body"] = df["body"].fillna("")

# --- 3-CLASS TARGET ---
def get_complexity(comments):
    if comments <= 1: return 0
    elif comments <= 5: return 1
    else: return 2

df["label"] = df["comments"].apply(get_complexity)
df["text"] = df["title"] + " " + df["body"]

print(f"Class Distribution: {df['label'].value_counts().to_dict()}")

# Split Data
train_df, val_df = train_test_split(
    df[["text", "label"]], 
    test_size=0.2, 
    stratify=df["label"], 
    random_state=RANDOM_STATE
)

# Convert to Hugging Face Datasets
hf_train = Dataset.from_pandas(train_df)
hf_val = Dataset.from_pandas(val_df)

# 2. Tokenization
print("Tokenizing data...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_train = hf_train.map(preprocess_function, batched=True)
tokenized_val = hf_val.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3. Define Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# 4. Initialize Model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

id2label = {0: "SIMPLE", 1: "MODERATE", 2: "COMPLEX"}
label2id = {"SIMPLE": 0, "MODERATE": 1, "COMPLEX": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)
model = model.to(device)

# 5. Define Training Arguments (UPDATED FOR EARLY STOPPING)
training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "bert_checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    report_to="none",
    
    # --- CHANGES FOR 10 EPOCHS & EARLY STOPPING ---
    num_train_epochs=10,              # We try for 10 epochs
    eval_strategy="epoch",            # Check score every epoch
    save_strategy="epoch",            # Save checkpoint every epoch
    load_best_model_at_end=True,      # Load the best one when finished
    metric_for_best_model="eval_loss",# We monitor Loss (or use "f1")
    greater_is_better=False           # Lower loss is better
)

# 6. Trainer (UPDATED WITH CALLBACK)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # --- ADDED CALLBACK ---
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 7. Train!
print("Starting Training...")
trainer.train()

# 8. Final Evaluation
print("\nFinal Evaluation on Validation Set:")
results = trainer.evaluate()
print(results)

# 9. Save
save_path = os.path.join(OUTPUT_DIR, "best_model_bert_3class")
print(f"Saving model to {save_path}...")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("DONE! Download the 'best_model_bert_3class' folder.")

2025-11-23 02:59:51.116500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763866791.291159     102 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763866791.344788     102 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_102/1155115568.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting Training...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.983000,0.935122,0.550033,0.490653,0.598320,0.550033


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [ ]:
# --- Install Gradient Boosting Libraries ---
!pip install -q xgboost lightgbm

In [4]:
import torch
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
from imblearn.over_sampling import SMOTE
import joblib
import torch.nn.functional as F

# --- CONFIGURATION ---
DATA_PATH = "/kaggle/input/github-issues-csv/github_issues.csv"
MODEL_PATH = "/kaggle/working/best_model_bert_3class" 
RANDOM_STATE = 42

print("=========================================================")
print("  PHASE 2 (REVISED): PROBABILITY STACKING + SMOTE  ")
print("=========================================================")

# 1. Load & Clean Data
df = pd.read_csv(DATA_PATH)

# Deduplication
if 'id' in df.columns:
    df = df.drop_duplicates(subset=['id'], keep='last')
else:
    df = df.drop_duplicates(subset=['title', 'body'], keep='last')

# Ensure columns
for col in ["title", "body", "comments", "reactions", "labels_count", "created_at"]:
    if col not in df.columns:
        if col in ("comments", "reactions", "labels_count"): df[col] = 0
        else: df[col] = ""

df["title"] = df["title"].fillna("").astype(str)
df["body"] = df["body"].fillna("").astype(str)
df["text"] = df["title"] + " " + df["body"]

# Target
def get_complexity(comments):
    if comments <= 1: return 0
    elif comments <= 5: return 1
    else: return 2
df["label"] = df["comments"].apply(get_complexity)

# 2. Generate Numeric Features
print("Generating numeric features...")
def has_pattern(text, pattern):
    return 1 if re.search(pattern, text, re.IGNORECASE) else 0

df['len_title'] = df['title'].apply(len)
df['len_body'] = df['body'].apply(len)
df['word_count'] = df['body'].apply(lambda x: len(x.split()))
df['has_code'] = df['body'].apply(lambda x: has_pattern(x, r'```|`[^`]+`'))
df['has_image'] = df['body'].apply(lambda x: has_pattern(x, r'!\[.*\]|<img'))
df['has_url'] = df['body'].apply(lambda x: has_pattern(x, r'http[s]?://'))

# User features
if 'user' in df.columns:
    df.sort_values('created_at', inplace=True)
    df['user_issue_count'] = df.groupby('user').cumcount()
    df['is_new_user'] = df['user_issue_count'].apply(lambda x: 1 if x == 0 else 0)
else:
    df['user_issue_count'] = 0; df['is_new_user'] = 1

numeric_cols = ['len_title', 'len_body', 'word_count', 'has_code', 'has_image', 'has_url', 
                'user_issue_count', 'is_new_user', 'labels_count', 'reactions']

X_numeric = df[numeric_cols].values
scaler = StandardScaler()
X_numeric = scaler.fit_transform(X_numeric)

# 3. Extract BERT Probabilities (The "Expert Opinion")
print(f"Extracting BERT probabilities from: {MODEL_PATH}...")

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)

def get_probabilities(text_list, batch_size=32):
    model.eval()
    all_probs = []
    for i in tqdm(range(0, len(text_list), batch_size), desc="Inference"):
        batch_text = text_list[i : i + batch_size]
        inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1).cpu().numpy() # Convert logits to 0-1 probs
            all_probs.append(probs)
    return np.vstack(all_probs)

X_bert_probs = get_probabilities(df['text'].tolist())
print(f"BERT Probs Shape: {X_bert_probs.shape}") # Should be (N, 3)

# 4. Combine Features [BERT Probs (3) + Numeric (10+)]
X_combined = np.hstack([X_bert_probs, X_numeric])
y_combined = df['label'].values

# 5. SMOTE & Training
print("Splitting and Balancing with SMOTE...")
X_train, X_val, y_train, y_val = train_test_split(X_combined, y_combined, test_size=0.2, random_state=RANDOM_STATE, stratify=y_combined)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"Training Set after SMOTE: {len(X_train_bal)} samples")

print("Training Meta-Model (XGBoost)...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,         # Shallower trees prevent overfitting
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    random_state=RANDOM_STATE,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

xgb_model.fit(X_train_bal, y_train_bal, eval_set=[(X_val, y_val)], verbose=100, early_stopping_rounds=50)

# 6. Evaluation
print("\n=== Final Evaluation (Probability Stacking + SMOTE) ===")
preds = xgb_model.predict(X_val)
print(classification_report(y_val, preds, target_names=["Simple", "Moderate", "Complex"]))
print(f"Accuracy: {accuracy_score(y_val, preds):.4f}")

# Feature Importance
print("\nFeature Importance:")
feature_names = ['BERT_Simple', 'BERT_Moderate', 'BERT_Complex'] + numeric_cols
for name, imp in sorted(zip(feature_names, xgb_model.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f"{name}: {imp:.4f}")

# Save
joblib.dump(xgb_model, "xgboost_stacking_model.joblib")
joblib.dump(scaler, "numeric_scaler.joblib")
print("Saved Stacking Model.")

  PHASE 2 (REVISED): PROBABILITY STACKING + SMOTE  
Generating numeric features...
Extracting BERT probabilities from: /kaggle/working/best_model_bert_3class...


Inference:   0%|          | 0/937 [00:00<?, ?it/s]

BERT Probs Shape: (29977, 3)
Splitting and Balancing with SMOTE...
Training Set after SMOTE: 32724 samples
Training Meta-Model (XGBoost)...


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[0]	validation_0-mlogloss:1.09150
[100]	validation_0-mlogloss:0.92312
[200]	validation_0-mlogloss:0.91537
[300]	validation_0-mlogloss:0.91265
[400]	validation_0-mlogloss:0.91148
[499]	validation_0-mlogloss:0.91055

=== Final Evaluation (Probability Stacking + SMOTE) ===
              precision    recall  f1-score   support

      Simple       0.63      0.62      0.63      2205
    Moderate       0.63      0.43      0.51      2727
     Complex       0.32      0.59      0.41      1064

    accuracy                           0.53      5996
   macro avg       0.53      0.55      0.52      5996
weighted avg       0.58      0.53      0.54      5996

Accuracy: 0.5290

Feature Importance:
BERT_Simple: 0.2728
BERT_Complex: 0.2255
BERT_Moderate: 0.1922
user_issue_count: 0.0688
len_title: 0.0394
has_url: 0.0381
len_body: 0.0355
word_count: 0.0351
has_code: 0.0337
has_image: 0.0318
is_new_user: 0.0273
labels_count: 0.0000
reactions: 0.0000
Saved Stacking Model.


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [04:05:34] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
